### Analysis Objective
The goal of this analysis is to identify **French ski lifts** that may be **under strain** due to **lack of snow**.

A ski lift is a transport system connecting two points in the mountains:
- **Downhill station (Aval)**: Starting point with specific geographic coordinates and altitude.
- **Uphill station (Amont)**: Endpoint with specific geographic coordinates and altitude.

---

### Required Data
To conduct this analysis, two main datasets are needed:

#### 1. Ski Lift Database
A script was developed to fetch the data from:
- [Remontées-Mécaniques.net](https://www.remontees-mecaniques.net/)
- French government open data ([data.gouv.fr](https://www.data.gouv.fr/))
- Data from the [National French Geographic Institute (IGN)](https://www.ign.fr/)

#### 2. Weather Data
A script was developed to:
- Calculate the **midpoint coordinates** of each ski lift's route (based on the downhill and uphill stations).
- Retrieve **daily snowfall data** for the standard ski season in France (**November 1, 2024 – April 30, 2025**) on [Open Méteo](https://open-meteo.com/).
- Store this information in a **JSON file**, linked to each ski lift's name.

---

### Data Processing with Spark
The processing pipeline includes the following steps:
1. **Data Cleaning**: Remove ski lifts that are **out of service** or demolished.
2. **Data Matching**: Associate snowfall data with each ski lift.
3. **Calculation**: Determine the **average snow cover** for each ski lift.
4. **Ranking**: Sort ski lifts by **snow cover level**.
5. **Export**: Generate a **CSV file** using Pandas.
6. **Visualization**: Create charts with Matplotlib for visual analysis.

---


# INSTALLING REQUIREMENT

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, avg, count, when, sum, max, min, regexp_replace, lag, desc, coalesce, expr, lit, trim
from pyspark.sql.window import Window


# SPARK INITIALIZATION

In [3]:
spark = SparkSession.builder \
    .appName("AnalyseNeigeRemontees") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "20") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .master("local[2]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f" Spark version: {spark.version}")
print(f" Hadoop version: {spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()}")


 Spark is ready!
 Spark version: 4.0.1
 Hadoop version: 3.4.1


# LOADING DATA

In [ ]:
df_mecaniques = spark.read.option("multiline", "true").json("ski-lift.json")
df_neige = spark.read.option("multiline", "true").json("ski-lift-snow-data.json")

print(f" Mechanical lifts: {df_mecaniques.count()} rows")
print(f" Snow data: {df_neige.count()} stations")

print("\n Data structure:")
print("Mechanical lifts:")
df_mecaniques.printSchema()
print("\nSnow data:")
df_neige.printSchema()

 Mechanical lifts: 2910 rows
 Snow data: 1978 stations

 Data structure:
Mechanical lifts:
root
 |-- altitude_amont: string (nullable = true)
 |-- altitude_aval: string (nullable = true)
 |-- annee_construction: string (nullable = true)
 |-- annee_fin_service: string (nullable = true)
 |-- appareil: string (nullable = true)
 |-- banniere: string (nullable = true)
 |-- capacite: string (nullable = true)
 |-- constructeur: string (nullable = true)
 |-- coordonnees: struct (nullable = true)
 |    |-- amont: struct (nullable = true)
 |    |    |-- lat: double (nullable = true)
 |    |    |-- lng: double (nullable = true)
 |    |-- aval: struct (nullable = true)
 |    |    |-- lat: double (nullable = true)
 |    |    |-- lng: double (nullable = true)
 |-- debit: string (nullable = true)
 |-- denivelee: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- donnees_manquantes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- emplacement_motrice: 

# CLEANING OF SKI LIFTS

In [8]:

# Device name harmonization
if "nom" in df_mecaniques.columns:
    df_mecaniques = df_mecaniques.withColumn("appareil", coalesce(col("nom"), col("appareil")))
elif "appareil" not in df_mecaniques.columns:
    print("  WARNING: Neither 'nom' nor 'appareil' found in the data!")
    df_mecaniques.show(5)

print("   Examples of raw altitudes:")
df_mecaniques.select("appareil", "station", "altitude_aval", "altitude_amont").show(5, truncate=False)

# Altitude cleaning (handles m, ' m', meter, meters)
df_mecaniques = df_mecaniques.withColumn(
    "altitude_aval_int",
    expr("""
        try_cast(
            regexp_replace(
                regexp_replace(
                    regexp_replace(altitude_aval, ' mètres?', ''),
                ' m', ''),
            'm', '') as int)
    """)
).withColumn(
    "altitude_amont_int",
    expr("""
        try_cast(
            regexp_replace(
                regexp_replace(
                    regexp_replace(altitude_amont, ' mètres?', ''),
                ' m', ''),
            'm', '') as int)
    """)
)

# Validation of altitude only if a value exists
df_mecaniques = df_mecaniques.withColumn(
    "altitude_aval_int",
    when(
        (col("altitude_aval").isNotNull()) & 
        (col("altitude_aval_int").isNotNull()) &
        (col("altitude_aval_int") > 0) &
        (col("altitude_aval_int") < 5000),
        col("altitude_aval_int")
    ).when(col("altitude_aval").isNull(), None)
     .otherwise(None)
).withColumn(
    "altitude_amont_int",
    when(
        (col("altitude_amont").isNotNull()) &
        (col("altitude_amont_int").isNotNull()) &
        (col("altitude_amont_int") > 0) &
        (col("altitude_amont_int") < 5000),
        col("altitude_amont_int")
    ).when(col("altitude_amont").isNull(), None)
     .otherwise(None)
)

# Calculation of average altitude
df_mecaniques = df_mecaniques.withColumn(
    "altitude_moyenne",
    when(
        col("altitude_aval_int").isNotNull() & col("altitude_amont_int").isNotNull(),
        (col("altitude_aval_int") + col("altitude_amont_int")) / 2
    ).when(
        col("altitude_aval_int").isNotNull(),
        col("altitude_aval_int")  # Use downhill if only downhill available
    ).when(
        col("altitude_amont_int").isNotNull(),
        col("altitude_amont_int")  # Use uphill if only uphill available
    ).otherwise(None)
)


nb_total_initial = df_mecaniques.count()
print(f"{'='*80}")
print(f"EXCLUSION")
print(f"{'='*80}")
print(f"Initial total lifts: {nb_total_initial}")

# Exclusion of lifts with end of service date
remontees_avec_fin_service = df_mecaniques.filter(
    col("annee_fin_service").isNotNull() & 
    (col("annee_fin_service") != "") &
    (trim(col("annee_fin_service")) != "")
)
nb_exclues_fin_service = remontees_avec_fin_service.count()
print(f"      Excluded (end of service date): {nb_exclues_fin_service}")
if nb_exclues_fin_service > 0:
    print("   Examples of lifts excluded for end of service date:")
    remontees_avec_fin_service.select("appareil", "station", "annee_fin_service").show(10, truncate=False)


df_apres_exclusion_fin_service = df_mecaniques.filter(
    (col("annee_fin_service").isNull()) | 
    (col("annee_fin_service") == "") | 
    (trim(col("annee_fin_service")) == "")
)
nb_apres_exclusion_fin_service = df_apres_exclusion_fin_service.count()
print(f"   Remaining lifts after end of service exclusion: {nb_apres_exclusion_fin_service}")

# Excluding lifts without altitude provided
remontees_sans_altitude = df_apres_exclusion_fin_service.filter(
    col("altitude_aval").isNull() & col("altitude_amont").isNull()
)
nb_sans_altitude = remontees_sans_altitude.count()
print(f"      Lifts without altitude (kept): {nb_sans_altitude}")

# Lifts with single altitude (downhill only)
remontees_altitude_aval_seule = df_apres_exclusion_fin_service.filter(
    col("altitude_aval_int").isNotNull() & col("altitude_amont_int").isNull()
)
nb_altitude_aval_seule = remontees_altitude_aval_seule.count()
print(f"   Kept (downhill altitude only): {nb_altitude_aval_seule}")

# Lifts with single altitude (uphill only)
remontees_altitude_amont_seule = df_apres_exclusion_fin_service.filter(
    col("altitude_aval_int").isNull() & col("altitude_amont_int").isNotNull()
)
nb_altitude_amont_seule = remontees_altitude_amont_seule.count()
print(f"   Kept (uphill altitude only): {nb_altitude_amont_seule}")

# Lifts with altitude provided but invalid (neither altitude can be converted to a valid number)
remontees_altitude_invalide = df_apres_exclusion_fin_service.filter(
    (col("altitude_aval").isNotNull() | col("altitude_amont").isNotNull()) &
    col("altitude_aval_int").isNull() &
    col("altitude_amont_int").isNull()
)
nb_altitude_invalide = remontees_altitude_invalide.count()
print(f"   Excluded (altitude provided but invalid/non-convertible): {nb_altitude_invalide}")
if nb_altitude_invalide > 0:
    print("   Examples of lifts excluded for invalid altitude:")
    remontees_altitude_invalide.select("appareil", "station", "altitude_aval", "altitude_amont", "altitude_moyenne").show(10, truncate=False)

# Lifts with valid altitude (one or two altitudes)
remontees_altitude_valide = df_apres_exclusion_fin_service.filter(
    col("altitude_moyenne").isNotNull()
)
nb_altitude_valide = remontees_altitude_valide.count()
print(f"   Kept (valid altitude - one or two): {nb_altitude_valide}")

# exclude only lifts with an end of service date
df_mecaniques_in_service = df_mecaniques.filter(
    (col("annee_fin_service").isNull()) | (col("annee_fin_service") == "") | (trim(col("annee_fin_service")) == "")
).filter(
    (col("altitude_aval").isNull() & col("altitude_amont").isNull()) |
    col("altitude_moyenne").isNotNull()
)

print(f" Retained lifts (except those with end of service date): {df_mecaniques_in_service.count()}")

print("\n   After cleaning:")
df_mecaniques_in_service.select(
    "appareil", 
    "station",
    "altitude_aval", 
    "altitude_aval_int",
    "altitude_amont",
    "altitude_amont_int",
    "altitude_moyenne"
).show(10, truncate=False)

stats = df_mecaniques_in_service.select(
    min("altitude_moyenne").alias("min_alt"),
    avg("altitude_moyenne").alias("moy_alt"),
    max("altitude_moyenne").alias("max_alt")
).collect()[0]

print(f"\n   Altitudes: Min={stats['min_alt']:.0f}m, Avg={stats['moy_alt']:.0f}m, Max={stats['max_alt']:.0f}m")

# Final exclusion summary
print(f"\n{'='*80}")
print("EXCLUSION SUMMARY")
print(f"{'='*80}")
print(f"Initial total: {nb_total_initial}")
print(f"Excluded (end of service date): {nb_exclues_fin_service}")
print(f"Excluded (invalid/non-convertible altitude): {nb_altitude_invalide}")
print(f"Kept (without altitude): {nb_sans_altitude}")
print(f"Kept (downhill altitude only): {nb_altitude_aval_seule}")
print(f"Kept (uphill altitude only): {nb_altitude_amont_seule}")
print(f"Kept (valid altitude - two altitudes): {nb_altitude_valide - nb_altitude_aval_seule - nb_altitude_amont_seule}")
print(f"Kept (valid altitude - total): {nb_altitude_valide}")
nb_total_conserves = df_mecaniques_in_service.count()
print(f"TOTAL KEPT: {nb_total_conserves}")
print(f"TOTAL EXCLUDED: {nb_total_initial - nb_total_conserves}")
print(f"{'='*80}\n")


   Examples of raw altitudes:
+--------------------+--------------------------------+-------------+--------------+
|appareil            |station                         |altitude_aval|altitude_amont|
+--------------------+--------------------------------+-------------+--------------+
|TCD6 de l'Essert    |Abondance (Les Portes du Soleil)|950 m        |1382 m        |
|TKD de la Pêche     |Abondance (Les Portes du Soleil)|994 m        |1041 m        |
|TKF du Covagny      |Abondance (Les Portes du Soleil)|1300 m       |1327 m        |
|TKD du Petit Frémoux|Abondance (Les Portes du Soleil)|1375 m       |1472 m        |
|TKD du Lac          |Abondance (Les Portes du Soleil)|1300 m       |1515 m        |
+--------------------+--------------------------------+-------------+--------------+
only showing top 5 rows
EXCLUSION
Initial total lifts: 2910
      Excluded (end of service date): 874
   Examples of lifts excluded for end of service date:
+------------------------------------------+----

# SNOW DATA PROCESSING
#### EXPLODING SNOW DATA TO GET DAILY RECORDS

In [9]:
df_neige_exploded = df_neige.select(
    col("station"),
    explode("historique_neige").alias("record")
).select(
    col("station"),
    col("record.date").alias("date"),
    col("record.snowfall_cm").alias("snowfall_cm")
).filter(col("snowfall_cm").isNotNull())

print(f"   Total raw measurements: {df_neige_exploded.count()}")

# Daily aggregation: sum of daily snowfall
print("\n   Aggregating daily snowfall")
df_neige_daily = df_neige_exploded.groupBy("station", "date").agg(
    sum("snowfall_cm").alias("snowfall_jour_cm")
).orderBy("station", "date")

print(f"   Unique days with measurements: {df_neige_daily.count()}")

# Calculation of cumulative snow cover (cumulative sum of snowfall)
print("\n   Calculating cumulative snow cover")
window_cumul = Window.partitionBy("station").orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

df_neige_clean = df_neige_daily.withColumn(
    "neige_cm",
    sum("snowfall_jour_cm").over(window_cumul)
).select(
    "station", 
    "date", 
    "neige_cm"
)

print(f"   Unique days with measurements: {df_neige_clean.count()}")
print("\n   Snow statistics (cumulative cm):")
stats = df_neige_clean.select(
    min("neige_cm").alias("min_cm"),
    avg("neige_cm").alias("moy_cm"),
    max("neige_cm").alias("max_cm")
).collect()[0]

print(f"   Min: {stats['min_cm']:.2f} cm")
print(f"   Average: {stats['moy_cm']:.2f} cm")
print(f"   Max: {stats['max_cm']:.2f} cm")

print("\n   Examples of cleaned data:")
df_neige_clean.orderBy("station", "date").show(10)

   Total raw measurements: 358018

   Aggregating daily snowfall
   Unique days with measurements: 45793

   Calculating cumulative snow cover
   Unique days with measurements: 45793

   Snow statistics (cumulative cm):
   Min: 0.00 cm
   Average: 1448.23 cm
   Max: 16810.90 cm

   Examples of cleaned data:
+--------------------+----------+--------+
|             station|      date|neige_cm|
+--------------------+----------+--------+
|Abondance (Les Po...|2024-11-01|     0.0|
|Abondance (Les Po...|2024-11-02|     0.0|
|Abondance (Les Po...|2024-11-03|     0.0|
|Abondance (Les Po...|2024-11-04|     0.0|
|Abondance (Les Po...|2024-11-05|     0.0|
|Abondance (Les Po...|2024-11-06|     0.0|
|Abondance (Les Po...|2024-11-07|     0.0|
|Abondance (Les Po...|2024-11-08|     0.0|
|Abondance (Les Po...|2024-11-09|     0.0|
|Abondance (Les Po...|2024-11-10|     0.0|
+--------------------+----------+--------+
only showing top 10 rows


#### CHECKING LIFTS WITHOUT SNOW DATA

In [10]:
# List of unique stations in each dataset
stations_mecaniques = df_mecaniques_in_service.select("station").distinct()
stations_neige = df_neige_clean.select("station").distinct()

print(f"Number of unique stations in mechanical lifts: {stations_mecaniques.count()}")
print(f"Number of unique stations in snow data: {stations_neige.count()}")

# Identification of stations without snow data
stations_sans_neige = stations_mecaniques.join(
    stations_neige,
    stations_mecaniques["station"] == stations_neige["station"],
    "left_anti"
)

print(f"\nNumber of stations WITHOUT snow data: {stations_sans_neige.count()}")

if stations_sans_neige.count() > 0:
    print("\nStations without snow data:")
    stations_sans_neige.show(20, truncate=False)
    
    remontees_sans_neige = df_mecaniques_in_service.alias("mec").join(
        stations_sans_neige.alias("stat"),
        col("mec.station") == col("stat.station"),
        "inner"
    ).select(col("mec.appareil"), col("mec.station")).distinct()
    
    print(f"\nNumber of lifts WITHOUT snow data: {remontees_sans_neige.count()}")
    print("\nExamples of lifts without snow data:")
    remontees_sans_neige.show(20, truncate=False)
else:
    print("\nAll stations have snow data")


Number of unique stations in mechanical lifts: 255
Number of unique stations in snow data: 253

Number of stations WITHOUT snow data: 6

Stations without snow data:
+------------------------+
|station                 |
+------------------------+
|Jougne - Champ aux Dames|
|Le Bouchet-Mont-Charvin |
|Chaume de Balveurche    |
|Saint-Firmin            |
|Le Collet - Retournemer |
|Mosset - Col de Jau     |
+------------------------+


Number of lifts WITHOUT snow data: 6

Examples of lifts without snow data:
+----------------------+------------------------+
|appareil              |station                 |
+----------------------+------------------------+
|TKD de la Savatte     |Le Bouchet-Mont-Charvin |
|TKF de Balveurche     |Chaume de Balveurche    |
|TKD du Col des Vachers|Saint-Firmin            |
|TKD de Jougne 2       |Jougne - Champ aux Dames|
|TKD du Col de Jau     |Mosset - Col de Jau     |
|TKD de Retournemer    |Le Collet - Retournemer |
+----------------------+--------------

# DATA JOINING

In [17]:
# Using a LEFT JOIN to include all lifts, even those without snow data
df_combine = df_mecaniques_in_service.alias("mec").join(
    df_neige_clean.alias("neige"),
    col("mec.station") == col("neige.station"),
    "left"
).select(
    col("mec.appareil"),
    col("mec.station"),
    col("mec.altitude_moyenne"),
    col("mec.altitude_aval_int"),
    col("mec.altitude_amont_int"),
    col("neige.date"),
    coalesce(col("neige.neige_cm"), lit(0.0)).alias("neige_cm")  # Replace NULL with 0.0
)

print(f"   Combined data: {df_combine.count()} rows")
print(f"   Lifts with data: {df_combine.select('appareil', 'station').distinct().count()}")

# Diagnostic of lifts without snow data
remontees_avec_neige = df_combine.filter(col("date").isNotNull()).select("appareil", "station").distinct()
remontees_sans_neige_count = df_mecaniques_in_service.count() - remontees_avec_neige.count()

if remontees_sans_neige_count > 0:
    print(f"\n       WARNING: {remontees_sans_neige_count} lifts do NOT have snow data!")

print("\n   Examples of combined data:")
df_combine.filter(col("date").isNotNull()).orderBy("station", "date").show(10)


   Combined data: 352956 rows
   Lifts with data: 1956


   Examples of combined data:
+--------------------+--------------------+----------------+-----------------+------------------+----------+--------+
|            appareil|             station|altitude_moyenne|altitude_aval_int|altitude_amont_int|      date|neige_cm|
+--------------------+--------------------+----------------+-----------------+------------------+----------+--------+
|     TKD de la Corne|Abondance (Les Po...|          1566.0|             1482|              1650|2024-11-01|     0.0|
|TKD du Grand Frémoux|Abondance (Les Po...|          1511.5|             1380|              1643|2024-11-01|     0.0|
|      TKE des Follys|Abondance (Les Po...|          1534.0|             1400|              1668|2024-11-01|     0.0|
|          TKD du Lac|Abondance (Les Po...|          1407.5|             1300|              1515|2024-11-01|     0.0|
|TKD du Petit Frémoux|Abondance (Les Po...|          1423.5|             1375|         

# RISK ANALYSES

In [13]:

df_combine_avec_neige = df_combine.filter(col("date").isNotNull())

df_moyenne = df_combine_avec_neige.groupBy("appareil", "station").agg(
    avg("neige_cm").alias("neige_moyenne_cm"),
    min("neige_cm").alias("neige_min_cm"),
    max("neige_cm").alias("neige_max_cm"),
    count("*").alias("nb_jours_mesure"),
    avg("altitude_moyenne").alias("altitude_moyenne")
)

# Identify lifts without snow data and add default values
remontees_avec_neige = df_moyenne.select("appareil", "station").distinct()
toutes_remontees = df_mecaniques_in_service.select("appareil", "station", "altitude_moyenne").distinct()
remontees_sans_neige = toutes_remontees.join(
    remontees_avec_neige,
    (toutes_remontees["appareil"] == remontees_avec_neige["appareil"]) & 
    (toutes_remontees["station"] == remontees_avec_neige["station"]),
    "left_anti"
).select(
    col("appareil"),
    col("station"),
    lit(0.0).alias("neige_moyenne_cm"),
    lit(0.0).alias("neige_min_cm"),
    lit(0.0).alias("neige_max_cm"),
    lit(0).alias("nb_jours_mesure"),
    col("altitude_moyenne")
)

# Combine lifts with and without snow data
df_moyenne = df_moyenne.union(remontees_sans_neige)

print("=== Top 10 lifts with the MOST snow ===")
df_moyenne.orderBy(desc("neige_moyenne_cm")).show(10, truncate=False)

print("=== Top 10 lifts with the LEAST snow ===")
df_moyenne.orderBy("neige_moyenne_cm").show(10, truncate=False)

# === CALCULATING RISK RATIOS ===
df_risque = df_combine_avec_neige.groupBy("appareil", "station").agg(
    (count(when(col("neige_cm") < 50, True)) / count("*")).alias("ratio_jours_faible_neige"),
    count(when(col("neige_cm") < 50, True)).alias("jours_neige_faible"),
    count(when(col("neige_cm") < 30, True)).alias("jours_neige_critique"),
    count("*").alias("jours_total")
)

# Add lifts without snow data with ratio = -1.0 (missing data)
remontees_sans_neige_risque = toutes_remontees.join(
    df_risque.select("appareil", "station").distinct(),
    (toutes_remontees["appareil"] == df_risque["appareil"]) & 
    (toutes_remontees["station"] == df_risque["station"]),
    "left_anti"
).select(
    col("appareil"),
    col("station"),
    lit(-1.0).alias("ratio_jours_faible_neige"),  # -1.0 for missing data
    lit(0).alias("jours_neige_faible"),
    lit(0).alias("jours_neige_critique"),
    lit(0).alias("jours_total")
)

# Combine lifts with and without snow data
df_risque = df_risque.union(remontees_sans_neige_risque)

print("=== Top 10 lifts at RISK (< 50 cm) ===")
df_risque.orderBy(desc("ratio_jours_faible_neige")).show(10, truncate=False)

print("Drought period detection")

window_sec = Window.partitionBy("appareil", "station").orderBy("date")

df_periode = df_combine_avec_neige.withColumn(
    "neige_insuffisante", 
    when(col("neige_cm") < 50, 1).otherwise(0)
)

df_periode = df_periode.withColumn(
    "changement",
    when(
        col("neige_insuffisante") != lag("neige_insuffisante", 1, 0).over(window_sec), 
        1
    ).otherwise(0)
)

df_periode = df_periode.withColumn(
    "groupe",
    sum("changement").over(window_sec.rowsBetween(Window.unboundedPreceding, 0))
)

df_secheresse = df_periode.filter(col("neige_insuffisante") == 1)     .groupBy("appareil", "station", "groupe").agg(
        count("*").alias("duree_jours"),
        min("date").alias("debut_periode"),
        max("date").alias("fin_periode"),
        avg("neige_cm").alias("neige_moy_periode")
    ).filter(col("duree_jours") >= 7)

print(f"\n   Total periods detected: {df_secheresse.count()}")
print("=== Top 10 longest drought periods ===")
df_secheresse.orderBy(desc("duree_jours")).show(10, truncate=False)

df_secheresse_agg = df_secheresse.groupBy("appareil", "station").agg(
    count("*").alias("nombre_periodes_secheresse"),
    max("duree_jours").alias("duree_max_secheresse"),
    avg("duree_jours").alias("duree_moyenne_secheresse")
)

# Add lifts without snow data with 0 drought periods
remontees_sans_secheresse = toutes_remontees.join(
    df_secheresse_agg.select("appareil", "station").distinct(),
    (toutes_remontees["appareil"] == df_secheresse_agg["appareil"]) & 
    (toutes_remontees["station"] == df_secheresse_agg["station"]),
    "left_anti"
).select(
    col("appareil"),
    col("station"),
    lit(0).alias("nombre_periodes_secheresse"),
    lit(0).alias("duree_max_secheresse"),
    lit(0.0).alias("duree_moyenne_secheresse")
)

# Combine lifts with and without drought periods
df_secheresse_agg = df_secheresse_agg.union(remontees_sans_secheresse)

print("=== Lifts with the most drought periods ===")
df_secheresse_agg.orderBy(desc("nombre_periodes_secheresse")).show(10, truncate=False)


=== Top 10 lifts with the MOST snow ===
+---------------------------+---------------------+----------------+------------+------------------+---------------+----------------+
|appareil                   |station              |neige_moyenne_cm|neige_min_cm|neige_max_cm      |nb_jours_mesure|altitude_moyenne|
+---------------------------+---------------------+----------------+------------+------------------+---------------+----------------+
|TSF6 des Quillis           |La Plagne (Paradiski)|9235.23038674033|0.0         |16571.700000000004|181            |NULL            |
|TSD6 de la Lovatière       |La Plagne (Paradiski)|9235.23038674033|0.0         |16571.700000000004|181            |NULL            |
|TSF4 de Plagne 1800        |La Plagne (Paradiski)|9235.23038674033|0.0         |16571.700000000004|181            |2019.0          |
|TSF4 des Mélèzes           |La Plagne (Paradiski)|9235.23038674033|0.0         |16571.700000000004|181            |1900.5          |
|TSF4 du Golf         

# FINAL RESULTS

In [14]:
# Create final DataFrame with ALL lifts (not just those with snow data)
df_coords_for_join = df_mecaniques_in_service.select(
    "appareil",
    "station",
    col("coordonnees.amont.lat").alias("latitude_amont"),
    col("coordonnees.amont.lng").alias("longitude_amont"),
    col("coordonnees.aval.lat").alias("latitude_aval"),
    col("coordonnees.aval.lng").alias("longitude_aval"),
    col("lien_reportage").alias("url_article"),
    col("altitude_moyenne")
).withColumn(
    "latitude",
    coalesce(col("latitude_amont"), col("latitude_aval"))
).withColumn(
    "longitude",
    coalesce(col("longitude_amont"), col("longitude_aval"))
).filter(
    col("latitude").isNotNull() & col("longitude").isNotNull()
).select(
    "appareil", 
    "station", 
    "latitude", 
    "longitude",
    "latitude_amont",
    "longitude_amont",
    "latitude_aval",
    "longitude_aval",
    "url_article",
    "altitude_moyenne"
)

# After coordinate filter

# Create base DataFrame with all lifts with coordinates
df_remontees_risque = df_coords_for_join

# Join with risk data (LEFT JOIN to keep all lifts)
df_remontees_risque = df_remontees_risque.join(
    df_risque,
    ["appareil", "station"],
    "left"
)

# Join with averages (LEFT JOIN) Exclude altitude_moyenne from df_moyenne as it's already in df_coords_for_join
df_moyenne_sans_altitude = df_moyenne.select(
    "appareil",
    "station",
    "neige_moyenne_cm",
    "neige_min_cm",
    "neige_max_cm",
    "nb_jours_mesure"
)
df_remontees_risque = df_remontees_risque.join(
    df_moyenne_sans_altitude,
    ["appareil", "station"],
    "left"
)

# Join with drought periods (LEFT JOIN)
df_remontees_risque = df_remontees_risque.join(
    df_secheresse_agg,
    ["appareil", "station"],
    "left"
)

# Replace NULL values with default values for lifts without snow data
df_remontees_risque = df_remontees_risque.fillna(0, subset=[
    "jours_neige_faible",
    "jours_neige_critique",
    "jours_total",
    "neige_moyenne_cm",
    "neige_min_cm",
    "neige_max_cm",
    "nb_jours_mesure",
    "nombre_periodes_secheresse",
    "duree_max_secheresse",
    "duree_moyenne_secheresse"
])


# Set ratio to -1.0 for missing data (if any)
df_remontees_risque = df_remontees_risque.fillna(-1.0, subset=["ratio_jours_faible_neige"])

print(f"   Lifts with snow data: {df_remontees_risque.filter(col('nb_jours_mesure').isNotNull() & (col('nb_jours_mesure') > 0)).count()}")
print(f"   Lifts without snow data: {df_remontees_risque.filter(col('nb_jours_mesure').isNull() | (col('nb_jours_mesure') == 0)).count()}")
print(f"   Lifts with coordinates: {df_remontees_risque.filter(col('latitude').isNotNull()).count()}")
print(f"   Lifts without coordinates: {df_remontees_risque.filter(col('latitude').isNull()).count()}")

print("\n" + "="*80)
print("TOP 20 LIFTS AT RISK")
print("="*80)
df_remontees_risque.select(
    "appareil",
    "station", 
    "neige_moyenne_cm",
    "altitude_moyenne",
    "nombre_periodes_secheresse"
).orderBy(desc("ratio_jours_faible_neige")).show(20, truncate=False)

   Lifts with snow data: 1896
   Lifts without snow data: 6
   Lifts with coordinates: 1902
   Lifts without coordinates: 0

TOP 20 LIFTS AT RISK
+-------------------------------------+---------------------------------+--------------------+----------------+--------------------------+
|appareil                             |station                          |neige_moyenne_cm    |altitude_moyenne|nombre_periodes_secheresse|
+-------------------------------------+---------------------------------+--------------------+----------------+--------------------------+
|ASC Panoramics                       |Langres                          |14.948066298342559  |NULL            |1                         |
|TKF Ecole                            |Noeux-les-Mines - Loisinord      |29.748066298342604  |68.0            |1                         |
|TSF4 de la Grande Découverte         |Carmaux - Cap' Découverte        |0.5906077348066308  |213.0           |1                         |
|FUNI 60  Évian-les-

# SAVE THE ANALYSIS

In [ ]:
# Count initial total
nb_total_initial = df_mecaniques.count()
nb_final = df_remontees_risque.count()

# Recalculate exclusions for summary
remontees_exclues_fin_service = df_mecaniques.filter(
    col("annee_fin_service").isNotNull() & 
    (col("annee_fin_service") != "") &
    (trim(col("annee_fin_service")) != "")
).count()

df_apres_fin_service = df_mecaniques.filter(
    (col("annee_fin_service").isNull()) | 
    (col("annee_fin_service") == "") | 
    (trim(col("annee_fin_service")) == "")
)

remontees_altitude_invalide = df_apres_fin_service.filter(
    (col("altitude_aval").isNotNull() | col("altitude_amont").isNotNull()) &
    col("altitude_aval_int").isNull() &
    col("altitude_amont_int").isNull()
).count()

remontees_sans_coords_final = df_mecaniques_in_service.filter(
    (col("coordonnees.amont.lat").isNull() & col("coordonnees.aval.lat").isNull()) |
    (col("coordonnees.amont.lng").isNull() & col("coordonnees.aval.lng").isNull()) |
    (col("coordonnees").isNull())
).count()

# Display some examples of each exclusion category
if remontees_exclues_fin_service > 0:
    print(f"\n   🔴 Excluded (end of service date) - Examples:")
    df_mecaniques.filter(
        col("annee_fin_service").isNotNull() & 
        (col("annee_fin_service") != "") &
        (trim(col("annee_fin_service")) != "")
    ).select("appareil", "station", "annee_fin_service").show(5, truncate=False)

if remontees_altitude_invalide > 0:
    print(f"\n   🟠 Excluded (invalid altitude) - Examples:")
    df_apres_fin_service.filter(
        (col("altitude_aval").isNotNull() | col("altitude_amont").isNotNull()) &
        col("altitude_moyenne").isNull()
    ).select("appareil", "station", "altitude_aval", "altitude_amont").show(5, truncate=False)

if remontees_sans_coords_final > 0:
    print(f"\n   🟡 Excluded (no coordinates) - Examples:")
    df_mecaniques_in_service.filter(
        (col("coordonnees.amont.lat").isNull() & col("coordonnees.aval.lat").isNull()) |
        (col("coordonnees.amont.lng").isNull() & col("coordonnees.aval.lng").isNull()) |
        (col("coordonnees").isNull())
    ).select("appareil", "station").show(5, truncate=False)

print(f"\n{'='*80}\n")

df_remontees_risque.toPandas().to_csv("data/analysis-results.csv", index=False)
print("Save successful")


   🔴 Excluded (end of service date) - Examples:
+---------------------------+------------------------------------------------+-----------------+
|appareil                   |station                                         |annee_fin_service|
+---------------------------+------------------------------------------------+-----------------+
|TKD de la Pêche            |Abondance (Les Portes du Soleil)                |2007             |
|TKF du Villard-dessus 1 & 2|Alex                                            |1976             |
|TSD4 du Lac Intrets        |Avoriaz (Morzine-Avoriaz - Les Portes du Soleil)|2023             |
|TSD4 du Proclou            |Avoriaz (Morzine-Avoriaz - Les Portes du Soleil)|2014             |
|TSF3 des Brochaux          |Avoriaz (Morzine-Avoriaz - Les Portes du Soleil)|2015             |
+---------------------------+------------------------------------------------+-----------------+
only showing top 5 rows

   🟠 Excluded (invalid altitude) - Examples:
+-------

# STOPPING SPARK

In [ ]:
spark.stop()